# Hi :)

----

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
import lightgbm as lgb

# ---------- Helpers ----------
def read_data(path):
    df = pd.read_csv(path, parse_dates=['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    return df

def add_basic_features(df):
    # assume df has Open, High, Low, Close, Adj Close, Volume
    df = df.copy()
    df['adj_return'] = df['Adj Close'].pct_change()  # daily return
    # log returns
    df['log_return'] = np.log(df['Adj Close'] / df['Adj Close'].shift(1)).fillna(0)
    # price ratios
    df['high_low'] = (df['High'] - df['Low']) / df['Low']
    df['close_open'] = (df['Close'] - df['Open']) / df['Open']
    # volume change
    df['vol_change'] = df['Volume'].pct_change().fillna(0)
    # time features
    df['dow'] = df['Date'].dt.dayofweek
    df['day'] = df['Date'].dt.day
    df['month'] = df['Date'].dt.month
    # lag features
    lags = [1,2,3,5,7,14,21]
    for l in lags:
        df[f'adj_ret_lag_{l}'] = df['adj_return'].shift(l)
        df[f'vol_lag_{l}'] = df['Volume'].shift(l)
    # rolling stats
    windows = [3,7,14,21,50]
    for w in windows:
        df[f'roll_mean_{w}'] = df['adj_return'].rolling(w).mean()
        df[f'roll_std_{w}'] = df['adj_return'].rolling(w).std()
        df[f'roll_vol_{w}'] = df['Volume'].rolling(w).mean()
    # fill/trim
    df.fillna(0, inplace=True)
    return df

def prepare_data(df):
    df = add_basic_features(df)
    # label is already in df['target']
    # drop columns that leak or not useful
    drop_cols = ['Date', 'Open','High','Low','Close','Adj Close']  # keep Volume maybe used
    features = [c for c in df.columns if c not in drop_cols + ['target']]
    return df, features

# ---------- Model training with time-aware CV ----------
def time_series_cv_train(df, features, n_splits=5):
    # We'll use expanding window: create splits by time blocks
    # Simpler: use GroupKFold on year-month groups (approx time blocks)
    df = df.reset_index(drop=True)
    df['year_month'] = df['Date'].dt.year * 100 + df['Date'].dt.month
    groups = df['year_month'].values
    gkf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(df), dtype=int)
    models = []
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(df, df['target'], groups)):
        X_tr = df.loc[tr_idx, features]
        y_tr = df.loc[tr_idx, 'target'].astype(int)
        X_val = df.loc[val_idx, features]
        y_val = df.loc[val_idx, 'target'].astype(int)

        lgb_train = lgb.Dataset(X_tr, label=y_tr)
        lgb_val = lgb.Dataset(X_val, label=y_val)

        params = {
            'objective': 'multiclass',
            'num_class': 5,
            'metric': 'None',
            'verbosity': -1,
            'learning_rate': 0.05,
            'num_leaves': 64,
            'min_data_in_leaf': 20,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'seed': 42
        }

        model = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_val],
            num_boost_round=2000,
            # early_stopping_rounds=100,
            # verbose_eval=False
        )
        models.append(model)
        preds = np.argmax(model.predict(X_val, num_iteration=model.best_iteration), axis=1)
        oof_preds[val_idx] = preds
        f1 = f1_score(y_val, preds, average='weighted')
        scores.append(f1)
        print(f'Fold {fold} F1-weighted: {f1:.4f}')

    overall_f1 = f1_score(df['target'].astype(int), oof_preds, average='weighted')
    print(f'OOF F1-weighted: {overall_f1:.4f}')
    return models, overall_f1

# ---------- Recursive forecasting for 365 days ----------
def recursive_forecast(df, models, features, days=365):
    # start from last available row
    df_work = df.copy().reset_index(drop=True)
    last_date = df_work['Date'].iloc[-1]
    preds = []
    for i in range(days):
        # build features for the next day
        next_date = last_date + pd.Timedelta(days=1)
        # naive row to hold features
        row = {}
        row['Date'] = next_date
        # For features that require historical data (lags/rolling), compute from df_work
        temp = df_work.copy()
        temp = add_basic_features(temp)  # ensures lag and rolling present
        last_row = temp.iloc[-1]
        # create new feature vector using last_row and computed lags (this is a simplified approach)
        # In practice recompute all features exactly as training pipeline did (consistent).
        for f in features:
            if f in last_row.index:
                row[f] = last_row[f]
            else:
                row[f] = 0
        X_next = pd.DataFrame([row])[features].fillna(0)
        # predict with ensembled models (average probabilities)
        probs = np.mean([m.predict(X_next, num_iteration=m.best_iteration)[0] for m in models], axis=0)
        pred_label = int(np.argmax(probs) - 2 + 2)  # careful: our labels are -2..2; ensure mapping below
        # Simpler: if training uses labels [-2,-1,0,1,2] encoded as 0..4, reverse map:
        pred_class = int(np.argmax(probs))  # 0..4
        mapped_label = pred_class - 2  # 0->-2, 1->-1, 2->0, 3->1, 4->2

        preds.append(mapped_label)

        # append a synthetic row to df_work so next iteration has history
        new_row = {
            'Date': next_date,
            'Open': np.nan, 'High': np.nan, 'Low': np.nan, 'Close': np.nan,
            'Adj Close': np.nan, 'Volume': 0, 'target': mapped_label
        }
        df_work = pd.concat([df_work, pd.DataFrame([new_row])], ignore_index=True)
        # ensure next iteration's last_date updated
        last_date = next_date
    return preds

# ---------- End-to-end ----------
def main(train_csv_path, out_csv='submission.csv'):
    df = read_data(train_csv_path)
    # If target not integer, ensure mapping to ints
    df['target'] = df['target'].astype(int)
    df, features = prepare_data(df)
    # encode labels into 0..4 for LightGBM
    label_map = {-2:0, -1:1, 0:2, 1:3, 2:4}
    df['target_enc'] = df['target'].map(label_map)
    # use target_enc for training
    df_for_train = df.copy()
    df_for_train['target'] = df_for_train['target_enc']

    models, oof_f1 = time_series_cv_train(df_for_train, features, n_splits=5)

    # recursive forecast
    preds = recursive_forecast(df[['Date','Open','High','Low','Close','Adj Close','Volume','target']], models, features, days=365)

    # save submission.csv
    submission = pd.DataFrame({'target': preds})
    submission.to_csv(out_csv, index=False)
    print(f'Saved {out_csv}')

main('data.csv', out_csv='submission.csv')


Fold 0 F1-weighted: 0.9982
Fold 1 F1-weighted: 0.9982
Fold 2 F1-weighted: 0.9964
Fold 3 F1-weighted: 0.9945
Fold 4 F1-weighted: 1.0000
OOF F1-weighted: 0.9974


C:\Users\Erfaan_Joodi\AppData\Local\Temp\ipykernel_15456\730302950.py:16: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['adj_return'] = df['Adj Close'].pct_change()  # daily return
C:\Users\Erfaan_Joodi\AppData\Local\Temp\ipykernel_15456\730302950.py:16: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['adj_return'] = df['Adj Close'].pct_change()  # daily return
C:\Users\Erfaan_Joodi\AppData\Local\Temp\ipykernel_15456\730302950.py:16: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA value

Saved submission.csv
